In [1]:
import boto3
from dotenv import load_dotenv
import os
import json

load_dotenv(".env")

aws_access_key_id = os.getenv("personal_AWS_ACCESS_KEY_ID")
aws_secret_access_key = os.getenv("personal_AWS_SECRET_ACCESS_KEY")

sesion_aws = boto3.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name="us-east-1"
)

# Create Bedrock Runtime and S3 Vectors clients in the AWS Region of your choice.
bedrock = sesion_aws.client("bedrock-runtime")
bedrock_agent = sesion_aws.client("bedrock-agent")
bedrock_agent_runtime = sesion_aws.client("bedrock-agent-runtime")
s3vectors = sesion_aws.client("s3vectors", verify=False)
s3_client = sesion_aws.client("s3")

kb_id = os.getenv("PERSONAL_KNOWLEDGE_BASE_ID")
model_arn = "arn:aws:bedrock:us-east-1::foundation-model/amazon.nova-lite-v1:0"

# Buckets S3

try:
    response = s3_client.list_buckets()
    print("Buckets S3 disponibles:")
    for bucket in response["Buckets"]:
        print(f"-- {bucket['Name']}")
except Exception as e:
    print(f"Error listando buckets: {str(e)}")


Buckets S3 disponibles:
-- bedrock-bda-us-east-1-d1778cf7-131d-4a3b-b651-1375e1f5440e
-- logs-buckets-01
-- test-inference-batch
-- test-source-s3-vector


In [5]:
# Parámetros del bucket y prefijo
bucket_name = "test-source-s3-vector"
prefix = "cars"

# Listar objetos en el bucket con el prefijo dado
response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

# Procesar cada archivo PDF
for obj in response.get("Contents", []):
    key = obj["Key"]
    if key.lower().endswith(".pdf"):
        print(f"Procesando: {key}")

        # Construir nombre del archivo de metadata
        metadata_key = key + ".metadata.json"

        # Crear contenido de metadata con nombre de archivo y KB_ID
        metadata = {
            "metadataAttributes": {
                "filename": key.split("/")[-1],
                "KB_ID": "CARS"
            }
        }

        valid = True
        for k, v in metadata["metadataAttributes"].items():
            size = len(v.encode("utf-8"))
            if size > 2048:
                print(f"Error: El atributo de metadata '{k}' excede el tamaño máximo permitido de 2048 bytes.")
                valid = False
                break

        # Subir el archivo de metadata a S3
        s3_client.put_object(
            Bucket=bucket_name,
            Key=metadata_key,
            Body=json.dumps(metadata).encode("utf-8"),
            ContentType="application/json"
        )

print("Archivos de metadata subidos exitosamente a S3.")

Procesando: cars/Como hacer un mantenimiento.pdf
Procesando: cars/MANUAL-DE-PARTES-CHEER.pdf
Procesando: cars/ford-centroamerica-bronco-2022-catalogo-descargable-esp.pdf
Procesando: cars/ford-centroamerica-bronco-sport-2021-catalogo-descargable-esp.pdf
Procesando: cars/ford-centroamerica-edge-2022-catalogo-descargable-esp.pdf
Procesando: cars/ford-centroamerica-mustang-2022-catalogo-descargable-esp.pdf
Archivos de metadata subidos exitosamente a S3.
